In [1]:
!pip install transformers scikit-learn pandas torch


[notice] A new release of pip is available: 24.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
import os, re, json, warnings
import numpy as np
import pandas as pd
import torch
import torch.nn.functional as F
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, f1_score, classification_report

warnings.filterwarnings("ignore")
torch.set_num_threads(os.cpu_count())   # use all CPU cores

print(f"CPU threads : {os.cpu_count()}")
print(f"Torch version: {torch.__version__}")

CPU threads : 8
Torch version: 2.11.0+cpu


In [3]:
DATA_PATH    = "mtsamples.csv"
ROBERTA_PATH = "roberta_best/hf_model"
BIOBERT_PATH = "BioBertModern/bioclinical_modernbert_specialty"

MIN_SAMPLES  = 20    # must match training
SEED         = 42    # must match training

# Quick sanity check — confirm all paths exist before doing anything else
for name, path in [("mtsamples.csv", DATA_PATH),
                   ("RoBERTa hf_model", ROBERTA_PATH),
                   ("BioBERT model", BIOBERT_PATH)]:
    exists = os.path.exists(path)
    status = "Yes" if exists else "No NOT FOUND"
    print(f"  {status}  {name}  →  {path}")

# Checking model.safetensors specifically Most Important
rob_sf = os.path.join(ROBERTA_PATH, "model.safetensors")
bio_sf = os.path.join(BIOBERT_PATH, "model.safetensors")
print()
print(f"  {'✓' if os.path.exists(rob_sf) else '✗ MISSING — download from Drive'}  roberta model.safetensors")
print(f"  {'✓' if os.path.exists(bio_sf) else '✗ MISSING'}  biobert model.safetensors")

  Yes  mtsamples.csv  →  mtsamples.csv
  Yes  RoBERTa hf_model  →  roberta_best/hf_model
  Yes  BioBERT model  →  BioBertModern/bioclinical_modernbert_specialty

  ✓  roberta model.safetensors
  ✓  biobert model.safetensors


In [4]:
def clean(text: str) -> str:
    """Exact same cleaner used in Roberta_finetuning.ipynb"""
    text = str(text)
    text = re.sub(r"\b\d{1,2}[/-]\d{1,2}[/-]\d{2,4}\b", " ", text)
    text = re.sub(r"\bMRN?[:\s#]*\d+\b", " ", text, flags=re.I)
    text = re.sub(r"\b\d{5,}\b", " ", text)
    text = re.sub(r"[^\w\s.,;:!?()-]", " ", text)
    text = re.sub(r"[ \t]+", " ", text).strip()
    return text

df = pd.read_csv(DATA_PATH)
df.drop(columns=[c for c in df.columns if "Unnamed" in c], inplace=True)
df["medical_specialty"] = df["medical_specialty"].str.strip()

counts = df["medical_specialty"].value_counts()
df["label"] = df["medical_specialty"].apply(
    lambda x: x if counts[x] >= MIN_SAMPLES else "Other"
)

# RoBERTa was trained on transcription + description fused
df["text_roberta"] = (
    df["transcription"].fillna("") + " " + df["description"].fillna("")
).apply(clean)

# BioBERT was trained on transcription only
df["text_biobert"] = df["transcription"].fillna("").apply(str)

df = df[df["text_roberta"].str.len() > 30].reset_index(drop=True)

print(f"Total rows : {len(df)}")
print(f"Classes    : {df['label'].nunique()}")
print()
print(df["label"].value_counts().to_string())

Total rows : 4987
Classes    : 30

label
Surgery                          1098
Consult - History and Phy.        516
Cardiovascular / Pulmonary        371
Orthopedic                        355
Radiology                         273
General Medicine                  259
Gastroenterology                  228
Neurology                         223
SOAP / Chart / Progress Notes     166
Obstetrics / Gynecology           160
Urology                           157
Other                             125
Discharge Summary                 108
ENT - Otolaryngology               96
Neurosurgery                       94
Hematology - Oncology              90
Ophthalmology                      83
Nephrology                         81
Emergency Room Reports             75
Pediatrics - Neonatal              70
Pain Management                    62
Psychiatry / Psychology            53
Office Notes                       50
Podiatry                           47
Dermatology                        29
Cosmetic 

In [5]:
le          = LabelEncoder()
y           = le.fit_transform(df["label"].values)
NUM_CLASSES = len(le.classes_)
id2label    = {int(i): c for i, c in enumerate(le.classes_)}

# Identical 70 / 15 / 15 stratified split — same random_state=42 as training
_, X_tmp_r, _, y_tmp = train_test_split(
    df["text_roberta"].values, y,
    test_size=0.30, stratify=y, random_state=SEED
)
_, X_tmp_b, _, _ = train_test_split(
    df["text_biobert"].values, y,
    test_size=0.30, stratify=y, random_state=SEED
)

X_te_r, _, y_te, _ = train_test_split(
    X_tmp_r, y_tmp,
    test_size=0.50, stratify=y_tmp, random_state=SEED
)
X_te_b, _, _, _ = train_test_split(
    X_tmp_b, y_tmp,
    test_size=0.50, stratify=y_tmp, random_state=SEED
)

print(f"Test samples : {len(y_te)}")
print(f"Classes      : {NUM_CLASSES}")
print(f"Class order  : {list(le.classes_[:4])} ...")

Test samples : 748
Classes      : 30
Class order  : ['Cardiovascular / Pulmonary', 'Consult - History and Phy.', 'Cosmetic / Plastic Surgery', 'Dentistry'] ...


In [6]:
print("Loading RoBERTa ")
rob_tok   = AutoTokenizer.from_pretrained(ROBERTA_PATH)
rob_model = AutoModelForSequenceClassification.from_pretrained(ROBERTA_PATH)
rob_model.eval()

with open(os.path.join(ROBERTA_PATH, "label_mappings.json")) as f:
    rob_id2label = json.load(f)["id2label"]   # keys are strings "0","1",...

print(f"  RoBERTa labels : {rob_model.config.num_labels}")
print()
print("Loading BioBERT weights into RAM ")
bio_tok   = AutoTokenizer.from_pretrained(BIOBERT_PATH)
bio_model = AutoModelForSequenceClassification.from_pretrained(BIOBERT_PATH)
bio_model.eval()

bio_id2label = {int(k): v for k, v in bio_model.config.id2label.items()}

print(f"  BioBERT labels : {bio_model.config.num_labels}")
print()
print("Both models loaded ✓")

Loading RoBERTa 


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

  RoBERTa labels : 30

Loading BioBERT weights into RAM 


Loading weights:   0%|          | 0/138 [00:00<?, ?it/s]

  BioBERT labels : 33

Both models loaded ✓


In [7]:
# Verify both models cover the exact same 40 specialties
bio_labels = set(bio_id2label.values())
rob_labels = set(rob_id2label.values())
only_bio   = bio_labels - rob_labels
only_rob   = rob_labels - bio_labels

print(f"Shared labels   : {len(bio_labels & rob_labels)}")
if only_bio:
    print(f"⚠  Only in BioBERT  : {only_bio}")
if only_rob:
    print(f"⚠  Only in RoBERTa  : {only_rob}")
if not only_bio and not only_rob:
    print("Label sets match perfectly ✓")

# Build reindex arrays so both probability vectors →
# canonical order (le.classes_, alphabetical)
rob_reindex = np.array([
    int(next(k for k, v in rob_id2label.items() if v == id2label[i]))
    for i in range(NUM_CLASSES)
])
bio_reindex = np.array([
    next(k for k, v in bio_id2label.items() if v == id2label[i])
    for i in range(NUM_CLASSES)
])

print("Reindex arrays built ✓")
print(f"Sample check — class 0 is '{id2label[0]}'")
print(f"  RoBERTa index for it : {rob_reindex[0]}")
print(f"  BioBERT index for it : {bio_reindex[0]}")

Shared labels   : 30
⚠  Only in BioBERT  : {'IME-QME-Work Comp etc.', 'Endocrinology', 'Bariatrics'}
Reindex arrays built ✓
Sample check — class 0 is 'Cardiovascular / Pulmonary'
  RoBERTa index for it : 0
  BioBERT index for it : 1


In [8]:
def get_probs(model, tokenizer, texts, max_len, batch_size=1):
    """
    Runs inference on all texts in batches.
    Returns numpy array of shape (N, num_labels).
    batch_size=1 is safest on CPU — increase
    """
    all_probs = []
    n = len(texts)

    for start in range(0, n, batch_size):
        batch = texts[start : start + batch_size].tolist()
        enc   = tokenizer(
            batch,
            max_length=max_len,
            padding=True,
            truncation=True,
            return_tensors="pt",
        )
        with torch.no_grad():
            probs = F.softmax(model(**enc).logits, dim=-1).numpy()
        all_probs.append(probs)

        # progress every 50 samples
        if start % 50 == 0:
            print(f"  {start:>4}/{n}", end="\r")

    print(f"  {n}/{n} ✓")
    return np.vstack(all_probs)   # (N, num_labels)


print("Running RoBERTa inference on 745 test samples...")
print("(the counter updates every 50 samples)\n")
rob_probs_raw = get_probs(rob_model, rob_tok, X_te_r, max_len=512)
rob_probs     = rob_probs_raw[:, rob_reindex]    # reorder to canonical

print(f"\nRoBERTa probs shape : {rob_probs.shape}")

Running RoBERTa inference on 745 test samples...
(the counter updates every 50 samples)

  748/748 ✓

RoBERTa probs shape : (748, 30)


In [9]:
print("Running BioBERT inference on 745 test samples...")
print("(~15–20 min on CPU — max_len=2048 so each sample takes longer)\n")
bio_probs_raw = get_probs(bio_model, bio_tok, X_te_b, max_len=2048)
bio_probs     = bio_probs_raw[:, bio_reindex]    # reorder to canonical

print(f"\nBioBERT probs shape : {bio_probs.shape}")
print("\nInference complete ✓")

Running BioBERT inference on 745 test samples...
(~15–20 min on CPU — max_len=2048 so each sample takes longer)

  748/748 ✓

BioBERT probs shape : (748, 30)

Inference complete ✓


In [10]:
# Predictions: argmax of each model's probability vector
rob_preds = rob_probs.argmax(axis=1)
bio_preds = bio_probs.argmax(axis=1)

# Ensemble: simple average of both distributions, then argmax
ens_probs = (rob_probs + bio_probs) / 2.0
ens_preds = ens_probs.argmax(axis=1)

results = {}
for name, preds in [("RoBERTa",  rob_preds),
                    ("BioBERT",  bio_preds),
                    ("Ensemble", ens_preds)]:
    results[name] = {
        "accuracy": accuracy_score(y_te, preds),
        "macro_f1": f1_score(y_te, preds, average="macro",    zero_division=0),
        "wtd_f1"  : f1_score(y_te, preds, average="weighted", zero_division=0),
    }

print(f"\n{'Model':<12} {'Accuracy':>10} {'Macro F1':>10} {'Weighted F1':>12}")
print("─" * 48)
for name, m in results.items():
    print(f"{name:<12} {m['accuracy']:>10.4f} {m['macro_f1']:>10.4f} {m['wtd_f1']:>12.4f}")

gain = results["Ensemble"]["macro_f1"] - max(
    results["RoBERTa"]["macro_f1"], results["BioBERT"]["macro_f1"]
)
print(f"\nEnsemble gain over best single model : {gain:+.4f}")


Model          Accuracy   Macro F1  Weighted F1
────────────────────────────────────────────────
RoBERTa          0.1872     0.1639       0.1760
BioBERT          0.3930     0.3456       0.3744
Ensemble         0.3864     0.3344       0.3674

Ensemble gain over best single model : -0.0112


In [11]:
print("=== Ensemble — Full Classification Report ===\n")
print(classification_report(y_te, ens_preds,
                             target_names=le.classes_,
                             zero_division=0))

=== Ensemble — Full Classification Report ===

                               precision    recall  f1-score   support

   Cardiovascular / Pulmonary       0.40      0.53      0.45        55
   Consult - History and Phy.       0.43      0.53      0.48        77
   Cosmetic / Plastic Surgery       0.00      0.00      0.00         4
                    Dentistry       0.38      0.75      0.50         4
                  Dermatology       0.33      0.40      0.36         5
            Discharge Summary       0.36      0.25      0.30        16
         ENT - Otolaryngology       0.38      0.43      0.40        14
       Emergency Room Reports       0.27      0.55      0.36        11
             Gastroenterology       0.48      0.62      0.54        34
             General Medicine       0.33      0.13      0.19        39
        Hematology - Oncology       0.29      0.15      0.20        13
                      Letters       1.00      0.25      0.40         4
                   Nephrology